<a href="https://colab.research.google.com/github/Damcharla/231FA04437-MLOps-Feast-SkillGap/blob/main/mlops_CLA_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import joblib


In [2]:
df = pd.read_csv("/content/titanic_pipeline_dataset.csv")

print("First 5 rows:")
print(df.head())

print("\nDataset Information:")
print(df.info())

First 5 rows:
   Pclass     Sex   Age  SibSp  Parch   Fare Embarked  Survived
0       2    male  17.7      0      1  35.70        Q         0
1       3    male  18.4      1      0  10.05        S         0
2       3  female  26.8      1      0  17.43        S         1
3       3  female  35.1      0      2  29.66        S         0
4       1  female  42.8      0      0  74.58        C         1

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    1000 non-null   int64  
 1   Sex       1000 non-null   object 
 2   Age       965 non-null    float64
 3   SibSp     1000 non-null   int64  
 4   Parch     1000 non-null   int64  
 5   Fare      980 non-null    float64
 6   Embarked  985 non-null    object 
 7   Survived  1000 non-null   int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 62.6+ KB
None


In [3]:
X = df[
    [
        "Pclass",
        "Sex",
        "Age",
        "SibSp",
        "Parch",
        "Fare",
        "Embarked"
    ]
]

y = df["Survived"]

print("\nFeatures:")
print(X.head())

print("\nTarget:")
print(y.head())


Features:
   Pclass     Sex   Age  SibSp  Parch   Fare Embarked
0       2    male  17.7      0      1  35.70        Q
1       3    male  18.4      1      0  10.05        S
2       3  female  26.8      1      0  17.43        S
3       3  female  35.1      0      2  29.66        S
4       1  female  42.8      0      0  74.58        C

Target:
0    0
1    0
2    1
3    0
4    1
Name: Survived, dtype: int64


In [4]:
numerical_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]

print("\nNumerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)



Numerical Features:
['Age', 'SibSp', 'Parch', 'Fare']

Categorical Features:
['Pclass', 'Sex', 'Embarked']


In [5]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "missing_values",
            SimpleImputer(strategy="median")
        ),
        (
            "scaling",
            StandardScaler()
        )
    ]
)

print("\nNumerical pipeline created.")



Numerical pipeline created.


In [6]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "missing_values",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoding",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

print("Categorical pipeline created.")

Categorical pipeline created.


In [7]:

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

print("\nPreprocessor created.")


Preprocessor created.


In [8]:
model_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(max_iter=1000)
        )
    ]
)

print("\nComplete ML pipeline created.")

print("""
Pipeline:
Raw Data
   ↓
Missing Value Handling
   ↓
Encoding
   ↓
Scaling
   ↓
Logistic Regression
   ↓
Survival Prediction
""")


Complete ML pipeline created.

Pipeline:
Raw Data
   ↓
Missing Value Handling
   ↓
Encoding
   ↓
Scaling
   ↓
Logistic Regression
   ↓
Survival Prediction



In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data size:", X_train.shape)
print("Testing data size:", X_test.shape)

Training data size: (800, 7)
Testing data size: (200, 7)


In [10]:
model_pipeline.fit(
    X_train,
    y_train
)

print("\nModel training completed.")


Model training completed.


In [11]:
predictions = model_pipeline.predict(X_test)

print("\nFirst 10 Predictions:")
print(predictions[:10])


First 10 Predictions:
[0 0 1 0 0 0 0 1 0 0]


In [14]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predictions
)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

Accuracy: 0.735
Accuracy Percentage: 73.5 %


In [15]:
new_passenger = pd.DataFrame(
    {
        "Pclass": [3],
        "Sex": ["male"],
        "Age": [25],
        "SibSp": [0],
        "Parch": [0],
        "Fare": [8.5],
        "Embarked": ["S"]
    }
)

prediction = model_pipeline.predict(
    new_passenger
)

print("\nNew Passenger:")
print(new_passenger)

print("\nPrediction:", prediction[0])

if prediction[0] == 1:
    print("Result: Passenger Survived")
else:
    print("Result: Passenger Did Not Survive")


New Passenger:
   Pclass   Sex  Age  SibSp  Parch  Fare Embarked
0       3  male   25      0      0   8.5        S

Prediction: 0
Result: Passenger Did Not Survive


In [16]:
joblib.dump(
    model_pipeline,
    "titanic_pipeline.pkl"
)

print("\nPipeline saved as titanic_pipeline.pkl")



Pipeline saved as titanic_pipeline.pkl


In [17]:
loaded_pipeline = joblib.load(
    "titanic_pipeline.pkl"
)

prediction = loaded_pipeline.predict(
    new_passenger
)

print("\nPrediction using loaded pipeline:", prediction[0])

if prediction[0] == 1:
    print("Result: Passenger Survived")
else:
    print("Result: Passenger Did Not Survive")


Prediction using loaded pipeline: 0
Result: Passenger Did Not Survive
